# Clotho-AQA Yes/No · Colab 실행

이 노트북은 **VS Code의 Google Colab 커널**에서 실행합니다. 로컬 코드를 고치는 작업은 VS Code에서, GPU 실행은 Colab에서 합니다. 모델은 공식 Qwen2.5-Omni-3B입니다. train/val/test 모두 질문 시작 단어에 관계없이 Yes/No 문항만 사용합니다. 학습 loss는 기존 답변 토큰 cross entropy를 유지합니다. 학습은 batch 64부터 GPU 메모리를 점검하고, 부족하면 32→16→8→4→2→1로 자동 낮춥니다. 배치를 낮추면 gradient accumulation으로 effective batch 약 64를 유지합니다.

1. 오른쪽 위 **Select Kernel → Colab → New Colab Server**에서 GPU 런타임을 고릅니다.
2. GPU 셀을 실행해 VRAM을 확인합니다. BF16과 약 40GB 이상의 GPU가 필요하며, 80GB를 권장합니다.
3. 아래 셀을 순서대로 실행합니다. Drive 마운트 코드와 학습 코드가 노트북 안에 포함되어 있어 별도 업로드가 필요 없습니다.

Colab 세션이 끝나면 `/content` 파일이 사라집니다. 공식 데이터 ZIP과 결과 TSV·어댑터는 Drive에 저장합니다. 학습 `.py` 파일을 수정한 뒤에는 로컬에서 `python3 build_colab_notebook.py`를 실행해 노트북을 갱신하세요.


In [ ]:
import torch
from pathlib import Path
assert torch.cuda.is_available(), "Colab GPU 서버를 선택하세요"
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
assert torch.cuda.is_bf16_supported(), "현재 코드는 BF16 GPU가 필요합니다"
if torch.cuda.get_device_properties(0).total_memory < 35 * 2**30:
    raise RuntimeError("현재 구현의 Qwen2.5-Omni LoRA에는 VRAM이 부족할 가능성이 큽니다. A100 40GB 이상을 선택하세요.")


## Drive 마운트와 코드 준비

다음 셀 맨 위에서 Google Drive를 마운트하고 현재 학습 코드를 `/content/training`에 풉니다. SSH 폴더에는 확장의 **Upload to Colab** 메뉴가 나타나지 않아도 됩니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import base64, io, zipfile
from pathlib import Path
_project_bundle = '...'
PROJECT = Path('/content/training')
PROJECT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_project_bundle))) as _archive:
    _archive.extractall(PROJECT)
assert (PROJECT / 'qwen_train.py').is_file()
assert Path('/content/drive/MyDrive').is_dir(), 'Google Drive 마운트를 확인하세요'
DRIVE = Path('/content/drive/MyDrive/clotho_aqa_omni')
DRIVE.mkdir(parents=True, exist_ok=True)
print('Code:', PROJECT, 'Persistent storage:', DRIVE)


In [ ]:
import subprocess, runpy, sys
# Colab may include an old optional torchao; PEFT rejects it even though this run uses plain LoRA.
subprocess.run(['python', '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
subprocess.run(['python', '-m', 'pip', 'install', '-r', str(PROJECT / 'requirements.txt')], check=True)
subprocess.run(['python', '-c', 'import importlib.util, torch, transformers, peft, swift, librosa, tqdm; assert importlib.util.find_spec("torchao") is None, "Remove the incompatible optional torchao package"; print(torch.__version__, transformers.__version__, peft.__version__)'], check=True)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
def run_script(script, *args):
    old_argv = sys.argv
    sys.argv = [str(PROJECT / script), *args]
    try:
        runpy.run_path(sys.argv[0], run_name='__main__')
    finally:
        sys.argv = old_argv
        import gc, torch
        gc.collect()
        torch.cuda.empty_cache()


## 공식 데이터 준비와 train 증강

공식 Clotho-AQA 데이터를 내려받고 train/val/test 모두 질문 시작 단어에 관계없이 Yes/No 문항만 남깁니다. train에 −30/−20/−10/+10/+20 dB 변형을 추가합니다. RMS 하한은 −65 dBFS이며, ±1 초과값도 클리핑 없이 32-bit float WAV에 저장합니다. 각 원본 문항에 무음/No도 하나씩 추가해 총 7배로 만듭니다. val/test는 원본을 유지합니다. ZIP과 CSV는 Drive에 보관하며 증강 WAV는 `/content/training/data/processed/augmented_audio/train`에 생성합니다.


In [ ]:
import hashlib, shutil, subprocess, zipfile
SOURCE = DRIVE / 'official_source'
SOURCE.mkdir(parents=True, exist_ok=True)
urls = {name: f'https://zenodo.org/records/6473207/files/{name}?download=1' for name in
        ['audio_files.zip', 'clotho_aqa_train.csv', 'clotho_aqa_val.csv', 'clotho_aqa_test.csv']}
for name, url in urls.items():
    target = SOURCE / name
    if not target.exists():
        subprocess.run(['curl', '-fL', '--retry', '3', url, '-o', str(target)], check=True)
archive = SOURCE / 'audio_files.zip'
with archive.open('rb') as stream:
    digest = hashlib.file_digest(stream, 'md5').hexdigest()
assert digest == '48c2c46ccd71dc06121228e2ed68d6b9', '공식 audio ZIP MD5가 다릅니다'
AUDIO = PROJECT / 'data/audio'
AUDIO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as zf:
    wavs = [name for name in zf.namelist() if name.lower().endswith('.wav')]
    assert len(wavs) == len(set(Path(name).name for name in wavs)), '중복 오디오 파일명이 있습니다'
    for name in wavs:
        target = AUDIO / Path(name).name
        if not target.exists():
            with zf.open(name) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst)
print('Extracted audio:', len(wavs))
_prepared = subprocess.run(['python', str(PROJECT/'clotho_data.py'), '--csv-dir', str(SOURCE),
                '--audio-dir', str(AUDIO), '--output-dir', str(PROJECT/'data/processed')],
                capture_output=True, text=True)
print(_prepared.stdout)
if _prepared.returncode:
    raise RuntimeError('Clotho 데이터 준비 실패:\n' + _prepared.stderr)


## ms-swift + Optuna: 짧은 실험부터

A100 80GB 1장, BF16. 먼저 cell 8에서 원본 val baseline을 다시 측정하세요. ① 데이터/분할/라벨 점검 ② OOM·데이터 로더·checkpointing 속도 측정 ③ 32문항 과적합/라벨 마스크/무음 반응 검사 ④ 원본 문항마다 증강 7종 중 하나씩 추출 ⑤ 50 step마다 고정된 val 200문항 평가 ⑥ rank 16에서 LR 우선 Optuna 탐색 ⑦ 쉘·TensorBoard·Optuna Dashboard 모니터링 순서입니다.

LR 5e-5와 1e-4를 먼저 비교하고 세 번째 LR은 TPE가 고릅니다. 유망한 결과가 있으면 해당 LR로 rank 32를 한 번 더 비교합니다. 각 trial은 최대 150 optimizer step, warmup 최대 10 step, 100 step부터 MedianPruner가 낮은 성능의 trial을 중단합니다. alpha 16, dropout 0.05, weight decay 0.1, effective batch 64를 유지합니다. 전체 예산은 준비 작업 포함 약 6시간이며 검증과 마무리는 예산을 넘길 수 있습니다. 최종 후보 최대 2개는 전체 val로 다시 비교합니다. audio encoder와 원본 가중치는 고정하며 Thinker 텍스트 attention/MLP LoRA만 학습합니다. 새 결과는 `results/optuna_budgeted/`에 저장하고 기존 grid 결과는 보존합니다.

탐색 셀은 학습 시작 전에 TensorBoard와 Optuna Dashboard 링크를 표시합니다. Scalars에서 `train/loss`, `train/learning_rate`, `train/grad_norm`을 첫 step과 이후 10 optimizer step마다 확인할 수 있습니다. val accuracy·macro-F1은 50 step마다 기록합니다. 각 조합과 OOM batch 시도는 별도 run으로 표시됩니다. 그래프는 새로고침하거나 자동 갱신을 사용하세요. 원시 step 기록은 각 batch 폴더의 `step_metrics.jsonl`에 저장합니다.


In [ ]:
run_script('qwen_train.py', '--mode', 'baseline',
           '--data-dir', str(PROJECT/'data/processed'),
           '--output-dir', str(DRIVE/'results/baseline_3b'))


In [ ]:
from monitoring import start_monitoring
SEARCH_ROOT = DRIVE / 'results/optuna_budgeted'
monitor_links = start_monitoring(SEARCH_ROOT)

assert (DRIVE/'results/baseline_3b/val_predictions.metrics.json').is_file(), \
       '먼저 cell 8에서 3B baseline을 완료하세요'
run_script('tune.py', '--data-dir', str(PROJECT/'data/processed'),
           '--baseline-metrics', str(DRIVE/'results/baseline_3b/val_predictions.metrics.json'),
           '--output-dir', str(SEARCH_ROOT), '--trials', '3', '--max-steps', '150',
           '--eval-steps', '50', '--min-steps', '100', '--timeout-seconds', '21600')


## 최적 체크포인트의 test 평가

짧은 탐색과 전체 val 재평가가 끝나면 전체 val 최고 점수의 어댑터로 test를 한 번 평가합니다. 탐색에서 저장한 어댑터를 사용하며 별도로 재학습하지 않습니다. 결과는 `results/optuna_final/`에 저장됩니다.


In [ ]:
run_script('finalize.py', '--data-dir', str(PROJECT/'data/processed'),
           '--best-tsv', str(DRIVE/'results/optuna_budgeted/best.tsv'),
           '--output-dir', str(DRIVE/'results/optuna_final'))
